In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# =============================================================================
# CMIP6 Spatial Climatology Calculator
# Computes detrended monthly climatologies for multiple models and variables
# Keeps full spatial dims (month × lat × lon) — saves one NetCDF per model/variable.
# =============================================================================

# --- Standard Library ---
import gc
import os

# --- Scientific Stack ---
import numpy as np
import xarray as xr
import xesmf as xe
from scipy.signal import detrend

# --- Dask ---
from dask.distributed import Client, LocalCluster

# --- CMIP6 Catalog ---
import intake


In [12]:
# =============================================================================
# CONFIGURATION
# =============================================================================

START_YEAR  = "1991"
END_YEAR    = "2014"
DEPTH_LEVEL = 0          # 0 = surface; set to depth in metres for 3-D vars

# Output region of interest
LAT_MIN, LAT_MAX = -90, -30   # south of 50°S
LON_MIN, LON_MAX =   1, 361

# 1° output grid
LAT_OUT = np.arange(LAT_MIN, LAT_MAX, 1.0)
LON_OUT = np.arange(LON_MIN, LON_MAX, 1.0)

# Save directory
SAVE_DIR = "/glade/work/diegovar/climatologies_spatial1/"
os.makedirs(SAVE_DIR, exist_ok=True)

# Catalog path
CATALOG_URL = (
    "/glade/collections/cmip/catalog/intake-esm-datastore/"
    "catalogs/glade-cmip6.json"
)

LOCAL_DATA_DIR = "/glade/derecho/scratch/diegovar/cmip6"

MODEL_MEMBERS = {
    # ── Core Si* ensemble ─────────────────────────────────────────────────────
    
    #"CESM2":              "r1i1p1f1",
    #"CESM2-FV2":          "r1i1p1f1",   # same MARBL-BEC as CESM2; variant only
    #"CESM2-WACCM":        "r1i1p1f1",   # same MARBL-BEC as CESM2; variant only
    #"CESM2-WACCM-FV2":    "r1i1p1f1",   # same MARBL-BEC as CESM2; variant only
   # "GFDL-ESM4":          "r1i1p1f1",
    "IPSL-CM6A-LR":       "r1i1p1f1",
    "UKESM1-0-LL":        "r1i1p1f2",   # f2 required: f1 has aerosol forcing error
    "CNRM-ESM2-1":        "r1i1p1f2",   # f2 required: center-designated standard run
    # ── Partial Si* ───────────────────────────────────────────────────────────
   # "GISS-E2-1-G-CC":     "r1i1p1f1",   # CC variant; si & no3 confirmed available
  #  "NorESM2-LM":         "r1i1p1f1",   # si & no3 present; missing phydiat, chl
  #  "NorCPM1":            "r1i1p1f1",   # decadal model; si & no3 present; limited vars
    # ── Physical variables only (no si/no3 in available files) ───────────────
  #  "GFDL-CM4":           "r1i1p1f1",
  #  "MIROC-ES2L":         "r1i1p1f2",   # f2 confirmed; si/no3 unavailable on NCAR
  #  "MPI-ESM1-2-HR":      "r1i1p1f1",
    #"MPI-ESM1-2-LR":      "r1i1p1f1",
    #"MPI-ESM-1-2-HAM":    "r1i1p1f1",
  #  "NorESM2-MM":         "r1i1p1f1",
  #  "ACCESS-ESM1-5":      "r1i1p1f1",#
  #  "CanESM5":            "r1i1p1f1",
    # ── Excluded (insufficient variable coverage) ─────────────────────────────
 #   "GISS-E2-1-G":        "r1i1p1f1",   # only tos, mlotst, so available
 #   "IPSL-CM5A2-INCA":    "r1i1p1f1",   # only so available; CMIP5-era variant
 #   "IPSL-CM6A-LR-INCA":  "r1i1p1f1",   # only so available; exclude from analysis
}

# MODELS list derived from MODEL_MEMBERS so the two stay in sync automatically
MODELS = list(MODEL_MEMBERS.keys())

VARIABLES = ['no3','si','si*']#["tos", "phyc","si","no3","phydiat","phydiaz","bsi","phydiat_r", "si*"]

# Member IDs and grid labels to try (in order)
MEMBER_ID_LIST = [
    "r1i1p1f1",
    "r1i1p1f2",
]
GRID_LIST = ["gr", "gn"]

# Unit-conversion functions applied AFTER standardize_coords
UNIT_CONVERSIONS = {
    "phyc":    lambda da: da * 12 * 1000,          # molC/m³ → mgC/m³
    "phydiat": lambda da: da * 12 * 1000,
    "phydiaz": lambda da: da * 12 * 1000,
    "zooc":    lambda da: da * 12 * 1000,
    "zoocos":  lambda da: da * 12 * 1000,
    "chl":     lambda da: da * 1e6,                # kg/m³  → mg/m³
    "intpp":   lambda da: da * 60*60*24*365*12,    # molC/m²/s → gC/m²/yr
    "epc100":  lambda da: da * 31_536_000,
    "fgco2":   lambda da: da * 1000*3600*24*365 / (44/12),
    "spco2":   lambda da: da * 1000 / 101.325,
    "dfe":     lambda da: da * 1e6,
    "no3":     lambda da: da * 1e3,
    "si":      lambda da: da * 1e3,
}

# Variables that must be clipped to >= 0 before processing
NON_NEGATIVE_VARS = {"phyc", "chl", "no3", "si", "dfe", "dissic", "talk"}

# Models / variable combos known to be missing or broken — skip silently
SKIP_COMBOS = {

}

OPEN_KWARGS = dict(
    chunks={"time": 24},
    parallel=True,
    combine="by_coords",
    data_vars="minimal",
    coords="minimal",
    compat="override",
)


In [4]:
# =============================================================================
# DASK CLUSTER
# =============================================================================

def start_cluster():
    cluster = LocalCluster(
        n_workers=4,
        threads_per_worker=2,
        memory_limit="7GB",      # 4 × 7GB = 28GB, leaves headroom for OS
        dashboard_address=":8787",
    )
    client = Client(cluster)
    print(client)
    return client


In [5]:
# =============================================================================
# CATALOG HELPER
# =============================================================================

_CATALOG_CACHE = None

def _get_catalog():
    """Load the intake-ESM catalog once and cache it for the session."""
    global _CATALOG_CACHE
    if _CATALOG_CACHE is None:
        print(f"    [catalog] Loading intake catalog from {CATALOG_URL} ...")
        _CATALOG_CACHE = intake.open_esm_datastore(CATALOG_URL)
        print(f"    [catalog] Loaded — {len(_CATALOG_CACHE.df):,} entries")
    return _CATALOG_CACHE


def get_path(model, variable, data_type, experiment_id, activity_id):
    """
    Return file paths for the correct member_id / grid_label combo.

    The member ID is looked up from MODEL_MEMBERS — each model has one
    designated standard run based on modeling-center recommendations and
    confirmed data availability (see config cell).

    Search order:
      1. GLADE intake-ESM catalog  (primary)
      2. Local directory at LOCAL_DATA_DIR  (fallback)
    """
    # ------------------------------------------------------------------ #
    # 1. Catalog search — use the single correct member ID for this model  #
    # ------------------------------------------------------------------ #
    cat = _get_catalog()
    run = MODEL_MEMBERS.get(model, "r1i1p1f1")  # safe default if model not in dict
    print(f"    [member] {model} → {run}")

    for grid in GRID_LIST:
        subset_grid = cat.search(
            experiment_id=[experiment_id],
            table_id=data_type,
            variable_id=variable,
            source_id=model,
            member_id=run,
            activity_id=activity_id,
            grid_label=grid,
        )
        if not subset_grid.df.empty:
            return subset_grid.df["path"].tolist()

    # ------------------------------------------------------------------ #
    # 2. Local-directory fallback                                          #
    # ------------------------------------------------------------------ #
    return get_path_local(model, variable, data_type, experiment_id)


def get_path_local(model, variable, data_type, experiment_id):
    """
    Search LOCAL_DATA_DIR for NetCDF files that match the requested
    model / variable / experiment combination.

    Tries two common CMIP6 sub-directory layouts in order:

      Layout A (DRS-style, most common on GLADE scratch):
        <root>/<activity>/<institution>/<model>/<experiment>/
               <member>/<table>/<variable>/<grid>/<version>/*.nc

      Layout B (flatter, often used for personal downloads):
        <root>/<model>/<variable>/**/*.nc
        <root>/<variable>/<model>/**/*.nc

    Returns a sorted list of matching paths, or [] if nothing is found.
    """
    import glob

    # Use the designated member ID for this model (same as catalog search)
    run = MODEL_MEMBERS.get(model, "r1i1p1f1")

    # ---- Layout A: walk the full DRS tree under LOCAL_DATA_DIR ----------
    #  Pin the member_id to the correct run; wildcard only for parts we
    #  genuinely don't know (institution, grid_label, version).
    pattern_A = os.path.join(
        LOCAL_DATA_DIR,
        "*",            # activity_id  (e.g. CMIP, DCPP …)
        "*",            # institution_id
        model,
        experiment_id,
        run,            # ← specific member ID, not wildcard
        data_type,
        variable,
        "*",            # grid_label
        "*",            # version
        "*.nc",
    )

    # ---- Layout B-1: <root>/<model>/<variable>/**/*.nc ------------------
    pattern_B1 = os.path.join(LOCAL_DATA_DIR, model, variable, "**", "*.nc")

    # ---- Layout B-2: <root>/<variable>/<model>/**/*.nc ------------------
    pattern_B2 = os.path.join(LOCAL_DATA_DIR, variable, model, "**", "*.nc")

    # ---- Layout B-3: broad fallback -------------------------------------
    pattern_B3 = os.path.join(LOCAL_DATA_DIR, "**", "*.nc")

    for pattern in (pattern_A, pattern_B1, pattern_B2):
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            # For B1/B2 (no member_id in path), filter to the correct run
            if pattern in (pattern_B1, pattern_B2):
                hits = [p for p in hits if run in p]
            if hits:
                print(f"    [local fallback] {model} {run} — found {len(hits)} file(s) via: {pattern}")
                return hits

    # Last resort: broad search filtered by model + variable + member + experiment
    all_nc = glob.glob(pattern_B3, recursive=True)
    hits = [
        p for p in all_nc
        if model in p and variable in p and experiment_id in p and run in p
    ]
    if hits:
        print(f"    [local fallback – broad search] {model} {run} — found {len(hits)} file(s)")
        return sorted(hits)

    print(f"    [local fallback] NO files found for {model} / {run} / {variable}")
    return []   # truly nothing found

# =============================================================================
# COORDINATE / DEPTH HELPERS
# =============================================================================

# AFTER
def standardize_coords(ds, depth_level):
    """Rename non-standard coordinate names and select the requested depth."""
    if "nav_lat" in ds.coords:
        ds = ds.rename({"nav_lon": "lon", "nav_lat": "lat"})
    if "latitude" in ds.coords and "lat" not in ds.coords:
        ds = ds.rename({"longitude": "lon", "latitude": "lat"})
    for depth_dim in ("lev_partial", "olevel", "lev"):
        if depth_dim in ds.coords:
            ds = ds.sel({depth_dim: depth_level}, method="nearest")
            break

    # ── Normalize longitude to 0–360 ──────────────────────────────────────
    # Some models (e.g. IPSL, CNRM) use -180→180; others use 0→360.
    # Standardize everything to 0→360 before regridding so all models
    # land on the same target grid with no spatial offsets.
    if "lon" in ds.coords:
        lon = ds["lon"]
        if float(lon.min()) < 0:
            # Roll the dataset so longitude goes 0→360 instead of -180→180
            ds = ds.assign_coords(lon=(lon % 360))
            ds = ds.sortby("lon")

    return ds


# =============================================================================
# REGRIDDER CACHE
# =============================================================================

_REGRIDDER_CACHE: dict = {}

def get_regridder(ds_data, ds_target):
    """Return a cached xe.Regridder; create one if the grid is new."""
    lat_vals = ds_data.lat.values
    lon_vals = ds_data.lon.values
    lat_flat = lat_vals.ravel()
    lon_flat = lon_vals.ravel()

    key = (
        lat_vals.shape, lon_vals.shape,
        float(lat_flat[0]), float(lat_flat[-1]),
        float(lon_flat[0]), float(lon_flat[-1]),
    )

    if key not in _REGRIDDER_CACHE:
        ds_in = xr.Dataset({"lat": ds_data.lat, "lon": ds_data.lon})
        _REGRIDDER_CACHE[key] = xe.Regridder(
            ds_in, ds_target, "bilinear",
            periodic=True, ignore_degenerate=True,
        )
    return _REGRIDDER_CACHE[key]


# =============================================================================
# DATA LOADING HELPERS
# =============================================================================

def _open_var(model, var, data_type="Omon",
              experiment_id="historical", activity_id="CMIP"):
    """Open a single CMIP6 variable as a lazy xarray DataArray."""
    path = get_path(model, var, data_type, experiment_id, activity_id)
    if not path:
        return None
    ds = xr.open_mfdataset(path, **OPEN_KWARGS)
    ds = ds.sel(time=slice(START_YEAR, END_YEAR))
    ds = standardize_coords(ds, DEPTH_LEVEL)
    return ds[var]


def load_variable(model, var):
    """
    Load and preprocess a DataArray for *var* from *model*.
    Returns (DataArray, var_name) or (None, None) if data unavailable.
    Handles derived variables (taum, phydiat_r, si*) transparently.
    """
    if (model, var) in SKIP_COMBOS:
        print(f"    Skipping known missing combo: {model} / {var}")
        return None, None

    # ------------------------------------------------------------------ taum
    if var == "taum":
        da1 = _open_var(model, "tauuo")
        da2 = _open_var(model, "tauvo")
        if da1 is None or da2 is None:
            return None, None
        da = xr.apply_ufunc(
            np.sqrt, da1**2 + da2**2,
            dask="parallelized", output_dtypes=[da1.dtype],
        )
        da = da.assign_coords(lat=da1.lat, lon=da1.lon)
        return da, "taum"

    # ------------------------------------------------------------ phydiat_r
    if var == "phydiat_r":
        da1 = _open_var(model, "phydiat")
        da2 = _open_var(model, "phyc")
        if da1 is None or da2 is None:
            return None, None
        return da1 / da2, "phydiat_r"

    # ------------------------------------------------------------------ si*
    if var == "si*":
        da1 = _open_var(model, "si")
        da2 = _open_var(model, "no3")
        if da1 is None or da2 is None:
            return None, None
        return da1 * 1e3 - da2 * 1e3, "si*"

    # --------------------------------------------------------- standard var
    da = _open_var(model, var)
    if da is None:
        return None, None

    if var in NON_NEGATIVE_VARS:
        da = da.where(da >= 0)

    if var in UNIT_CONVERSIONS:
        da = UNIT_CONVERSIONS[var](da)

    return da, var



In [6]:
# =============================================================================
# SPATIAL CLIMATOLOGY COMPUTATION
# =============================================================================

def compute_spatial_climatology(da, ds_grid):
    """
    Given a lazy DataArray *da* on its native grid:
      1. Regrid to 1° target grid
      2. Subset to LAT_MIN – LAT_MAX / LON_MIN – LON_MAX
      3. Compute time mean and anomaly
      4. Detrend anomaly along time axis
      5. Compute detrended monthly climatology
    Returns xr.DataArray of shape (month=12, lat, lon).
    Single .compute() at the end — no intermediate loads.
    """
    # --- 1. Regrid (lazy) ---
    regridder = get_regridder(da, ds_grid)
    da_rg     = regridder(da)

    # --- 2. Subset to region of interest ---
    da_rg = da_rg.sel(
        lat=slice(LAT_MIN, LAT_MAX),
        lon=slice(LON_MIN, LON_MAX),
    )

    # --- 3. Time mean and anomaly (lazy) ---
    time_mean = da_rg.mean(dim="time", skipna=True)
    anom      = da_rg - time_mean

    # --- 4. Detrend — full time axis in one chunk; small spatial chunks ---
    anom_filled = anom.fillna(0).chunk({"time": -1, "lat": 10, "lon": 36})
    anom_detrended = xr.apply_ufunc(
        detrend,
        anom_filled,
        kwargs={"axis": 0},
        dask="parallelized",
        output_dtypes=[anom.dtype],
    ).where(~anom.isnull())

    detrended = anom_detrended + time_mean

    # --- 5. Monthly climatology — single .compute(), spatial dims preserved ---
    clim = (
        detrended
        .groupby("time.month")
        .mean("time", skipna=True)
        .compute()
    )
    return clim   # (month=12, lat, lon)


# =============================================================================
# SAVE / PATH HELPERS
# =============================================================================

def save_path(model, var):
    suffix = (f"{DEPTH_LEVEL}m_" if DEPTH_LEVEL != 0 else "")
    # Replace "si*" → "sistar" so the filename is shell-safe
    safe_var = var.replace("*", "star")
    return os.path.join(SAVE_DIR, f"{model}_{safe_var}_{suffix}{START_YEAR}{END_YEAR}_spatial.nc")


def save_spatial_clim(clim_da, model, var):
    path = save_path(model, var)
    safe_var = var.replace("*", "star")
    ds_out = clim_da.to_dataset(name=safe_var)
    ds_out.attrs.update({
        "model":      model,
        "variable":   var,
        "start_year": START_YEAR,
        "end_year":   END_YEAR,
        "region":     f"lat {LAT_MIN} to {LAT_MAX}, lon {LON_MIN} to {LON_MAX}",
    })
    ds_out.to_netcdf(path)
    print(f"    Saved → {path}")


# =============================================================================
# MAIN LOOP
# =============================================================================

def run():
    # 1° output grid — defined once, never overwritten
    ds_grid = xr.Dataset({
        "lat": (["lat"], LAT_OUT),
        "lon": (["lon"], LON_OUT),
    })

    for model in MODELS:
        print(f"\n{'='*60}")
        print(f"Model: {model}")
        print(f"{'='*60}")

        for var in VARIABLES:
            print(f"  Variable: {var}")

            # --- Skip if output already exists ---
            out_path = save_path(model, var)
            if os.path.exists(out_path) and not OVERWRITE:
                print("    Output exists — skipping.")
                continue

            if os.path.exists(out_path) and OVERWRITE:
                print(" Output exists — overwriting (OVERWRITE=True).")

            da = None
            try:
                # --- Load (lazy) ---
                da, var_name = load_variable(model, var)
                if da is None:
                    print("    No data found — skipping.")
                    continue

                # --- Compute spatial climatology ---
                clim = compute_spatial_climatology(da, ds_grid)

                # --- Save ---
                save_spatial_clim(clim, model, var)

            except Exception as e:
                print(f"    ERROR for {model}/{var}: {e}")

            finally:
                del da
                gc.collect()
                client.run(gc.collect)
                client.run(lambda: None)

    print("\nAll spatial climatologies done.")


In [13]:
# =============================================================================
# ENTRY POINT
# =============================================================================
OVERWRITE = True

if __name__ == "__main__":
    client = start_cluster()
    try:
        run()
    finally:
        client.close()

<Client: 'tcp://127.0.0.1:45453' processes=4 threads=8, memory=26.08 GiB>

Model: IPSL-CM6A-LR
  Variable: no3
 Output exists — overwriting (OVERWRITE=True).
    [member] IPSL-CM6A-LR → r1i1p1f1
    ERROR for IPSL-CM6A-LR/no3: Input DataArray is not 1-D.
  Variable: si
 Output exists — overwriting (OVERWRITE=True).
    [member] IPSL-CM6A-LR → r1i1p1f1
    ERROR for IPSL-CM6A-LR/si: Input DataArray is not 1-D.
  Variable: si*
 Output exists — overwriting (OVERWRITE=True).
    [member] IPSL-CM6A-LR → r1i1p1f1
    ERROR for IPSL-CM6A-LR/si*: Input DataArray is not 1-D.

Model: UKESM1-0-LL
  Variable: no3
    [member] UKESM1-0-LL → r1i1p1f2


2026-03-24 07:15:13,317 - distributed.worker - ERROR - Compute Failed
Key:       open_dataset-6a8b0bae-eae6-419e-b144-57028ae463af
State:     executing
Task:  <Task 'open_dataset-6a8b0bae-eae6-419e-b144-57028ae463af' open_dataset(..., ...)>
Exception: "OSError(-101, 'NetCDF: HDF error')"
Traceback: '  File "/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/xarray/backends/api.py", line 606, in open_dataset\n    backend_ds = backend.open_dataset(\n        filename_or_obj,\n    ...<2 lines>...\n        **kwargs,\n    )\n  File "/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/xarray/backends/netCDF4_.py", line 767, in open_dataset\n    store = NetCDF4DataStore.open(\n        filename_or_obj,\n    ...<8 lines>...\n        autoclose=autoclose,\n    )\n  File "/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/xarray/backends/netCDF4_.py", line 525, in open\n    return cls(manager, group=group, mode=mode, lock=lock, autoclose

    ERROR for UKESM1-0-LL/no3: [Errno -101] NetCDF: HDF error: '/glade/collections/cmip/CMIP6/CMIP/MOHC/UKESM1-0-LL/historical/r1i1p1f2/Omon/no3/gn/v20190627/no3/no3_Omon_UKESM1-0-LL_historical_r1i1p1f2_gn_195001-199912.nc'
  Variable: si
 Output exists — overwriting (OVERWRITE=True).
    [member] UKESM1-0-LL → r1i1p1f2
    ERROR for UKESM1-0-LL/si: Input DataArray is not 1-D.
  Variable: si*
    [member] UKESM1-0-LL → r1i1p1f2
    ERROR for UKESM1-0-LL/si*: Input DataArray is not 1-D.

Model: CNRM-ESM2-1
  Variable: no3
 Output exists — overwriting (OVERWRITE=True).
    [member] CNRM-ESM2-1 → r1i1p1f2
    ERROR for CNRM-ESM2-1/no3: Input DataArray is not 1-D.
  Variable: si
 Output exists — overwriting (OVERWRITE=True).
    [member] CNRM-ESM2-1 → r1i1p1f2
    ERROR for CNRM-ESM2-1/si: Input DataArray is not 1-D.
  Variable: si*
 Output exists — overwriting (OVERWRITE=True).
    [member] CNRM-ESM2-1 → r1i1p1f2
    ERROR for CNRM-ESM2-1/si*: Input DataArray is not 1-D.

All spatial clim